In [ ]:
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# ---------------------------------------------------
# 1. Load sample data
# ---------------------------------------------------
df = spark.table("samples.bakehouse.sales_transactions").toPandas()

print("Dataset shape:", df.shape)
print(df.head())

# ---------------------------------------------------
# 2. Feature engineering
# ---------------------------------------------------
df["dateTime"] = pd.to_datetime(df["dateTime"])

df["hour"] = df["dateTime"].dt.hour
df["day_of_week"] = df["dateTime"].dt.dayofweek


# Features and target
target = "totalPrice"

features = [
    "quantity",
    "unitPrice",
    "hour",
    "day_of_week",
    "paymentMethod",
    "product"
]
# Remove missing values
df = df.dropna(subset=features + [target])

X = df[features]
y = df[target]

# Optional: sample for faster execution
if len(df) > 20000:
    sampled_idx = df.sample(20000, random_state=42).index
    X = X.loc[sampled_idx]
    y = y.loc[sampled_idx]

# ---------------------------------------------------
# 3. Train/test split
# ---------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ---------------------------------------------------
# 4. Preprocessing
# ---------------------------------------------------
categorical_features = ["paymentMethod", "product"]
numeric_features = ["quantity", "unitPrice", "hour", "day_of_week"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)


# ---------------------------------------------------
# 5. Model pipeline
# ---------------------------------------------------
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ))
])


# ---------------------------------------------------
# 6. Enable MLflow autologging
# ---------------------------------------------------
mlflow.autolog()

# ---------------------------------------------------
# 7. Train model
# ---------------------------------------------------
with mlflow.start_run(run_name="bakehouse_total_price_prediction"):
    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    mse = mean_squared_error(y_test, preds)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, preds)

    # Explicit test metrics
    mlflow.log_metric("test_rmse", rmse)
    mlflow.log_metric("test_r2", r2)

    print(f"RMSE: {rmse:.2f}")
    print(f"R²: {r2:.4f}")

print("Training complete")